# Week 9: C traceback checks and baseline timings (Colab)

Run on a **GPU runtime** (Runtime > Change runtime type > T4 GPU) so the session matches the CUDA
timings taken later. **Run all cells top to bottom in one session.** The official baseline times come
from this notebook, not from anyone's laptop.

1. environment record, 2. clone, 3. build, 4. correctness (C vs Python), 5. sanitizer, 6. timings,
7. download the results and commit them to `results/`.

## 1. Environment (record this with the timings)

In [ ]:
%%bash
echo "--- GPU";  nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo "no GPU"
echo "--- CPU";  lscpu | grep -E "Model name|^CPU\(s\)|Thread|MHz" 
echo "--- gcc";  gcc --version | head -1
echo "--- python"; python3 --version
echo "--- date"; date

## 2. Get the repository

Edit `BRANCH` if you test another branch. The cell deletes and re-clones `sw_repo` every time.

In [ ]:
import glob, os, subprocess

REPO_URL = "https://github.com/Luis-AQR/CEPARCO_GRP3_IntegratingProject.git"
BRANCH   = "main"

os.chdir("/content")
subprocess.run("rm -rf sw_repo", shell=True, check=True)
subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, "sw_repo"], check=True)

# the project root is the folder that contains c/sw_seq.c (smith-waterman/ or the repo root)
hits = glob.glob("/content/sw_repo/**/c/sw_seq.c", recursive=True)
assert hits, "c/sw_seq.c not found - wrong branch?"
ROOT = os.path.dirname(os.path.dirname(hits[0]))
os.chdir(ROOT)
print("project root:", ROOT)
print(subprocess.run("git log --oneline -3 && ls", shell=True, capture_output=True, text=True).stdout)

## 3. Build (same flags as the Windows build, `-O2`, no extra optimisation)

In [ ]:
%%bash
mkdir -p bin
gcc -std=c99 -O2 -Wall -Wextra -o bin/sw_seq c/sw_seq.c && echo "built bin/sw_seq (no warnings above = clean)"

## 4. Correctness: C vs the independent Python reference

Both must end with `All C results match the reference.` `--all` includes `bench_5000` (pure Python,
takes a minute or two).

In [ ]:
%%bash
set -e
python3 scripts/check_expected.py | tail -3
python3 scripts/check_c_week8.py --all
python3 scripts/check_c_week9.py --all

## 5. Memory safety: AddressSanitizer + UndefinedBehaviorSanitizer

Not timed (sanitizers slow the program down). Any report from the sanitizers fails the cell.

In [ ]:
%%bash
set -eo pipefail
gcc -std=c99 -g -O1 -fsanitize=address,undefined -fno-omit-frame-pointer -o bin/sw_asan c/sw_seq.c
for t in test_paper test_identical test_nomatch test_gap bench_1000; do
  echo "== $t"
  ./bin/sw_asan data/$t.fasta --no-matrix --repeat 2 --warmup | grep -E "Max score|Re-score|No local alignment"
done
echo
echo "sanitizer run finished: no reports"

## 6. Baseline timings

Warm-up run + 5 timed runs for 1,000 and 5,000, warm-up + 3 for 10,000, each with and without the
traceback. Every run must reproduce the verified score, cell and checksum or the script stops.
Writes `results/week9_baseline.csv` and `results/week9_baseline.md`.

In [ ]:
%%bash
python3 scripts/run_baseline.py --exe bin/sw_seq

## 7. Save the results

Download the two files, copy them into the repository's `results/` folder, and commit them. They are
the official Week 9 baseline. Do not commit results from a session that was not on Colab.

In [ ]:
import os, shutil
from google.colab import files

shutil.make_archive("/content/week9_results", "zip", "results")
files.download("/content/week9_results.zip")